# Notebook 29 — Sparse Feature Emergence at the Reset Boundary

Notebook 29 studies sparse transition behavior near the modulo-30 reset path:

23 → 29 → 01

This notebook treats lane 29 as the final admissible residue lane before reset into lane 01.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import zscore
import json


In [ ]:
FIG_DIR = Path("figures")
RES_DIR = Path("results")
REP_DIR = Path("reports")

for d in [FIG_DIR, RES_DIR, REP_DIR]:
    d.mkdir(exist_ok=True)


In [ ]:
MODULUS = 30
LANES = [1,7,11,13,17,19,23,29]

def prime_sieve(n):
    sieve = np.ones(n + 1, dtype=bool)
    sieve[:2] = False

    for i in range(2, int(np.sqrt(n)) + 1):
        if sieve[i]:
            sieve[i*i::i] = False

    return np.where(sieve)[0]

primes = prime_sieve(2_000_000)

prime_residues = primes % MODULUS
prime_residues = prime_residues[np.isin(prime_residues, LANES)]

len(prime_residues)


## Rolling lane windows


In [ ]:
WINDOW = 400
STEP = 40

rows = []

for start in range(0, len(prime_residues) - WINDOW, STEP):

    block = prime_residues[start:start + WINDOW]

    counts = {}

    for lane in LANES:
        counts[f"{lane:02d}"] = int(np.sum(block == lane))

    rows.append(counts)

df = pd.DataFrame(rows)

df.head()


## Reset-boundary features


In [ ]:
df["gap_23_29"] = np.abs(df["23"] - df["29"])
df["gap_29_01"] = np.abs(df["29"] - df["01"])

df["reset_pressure"] = (
    df["gap_29_01"] / (df["29"] + df["01"])
)

df["boundary_pressure"] = (
    df["gap_23_29"] / (df["23"] + df["29"])
)

df.head()


## Sparse-event detection


In [ ]:
df["z_29"] = zscore(df["29"])
df["z_gap_29_01"] = zscore(df["gap_29_01"])
df["z_reset_pressure"] = zscore(df["reset_pressure"])

SPARSE_THRESHOLD = 2.0

df["sparse_event"] = (
    (np.abs(df["z_29"]) > SPARSE_THRESHOLD)
    |
    (np.abs(df["z_gap_29_01"]) > SPARSE_THRESHOLD)
    |
    (np.abs(df["z_reset_pressure"]) > SPARSE_THRESHOLD)
).astype(int)

df["sparse_event"].value_counts()


## Reset-state classification


In [ ]:
states = []

for _, row in df.iterrows():

    if row["29"] > row["01"] + 5:
        states.append("29-leading")

    elif row["01"] > row["29"] + 5:
        states.append("01-leading")

    elif row["23"] > row["29"] + 5:
        states.append("23-leading")

    elif row["reset_pressure"] > 0.12:
        states.append("reset-gap")

    else:
        states.append("balanced")

df["state"] = states

df["state"].value_counts()


## Reset-boundary lane trajectories


In [ ]:
plt.figure(figsize=(14,6))

plt.plot(df["23"], label="23")
plt.plot(df["29"], label="29")
plt.plot(df["01"], label="01")

plt.title("Notebook 29 — Reset Boundary Lane Counts")
plt.xlabel("Rolling window")
plt.ylabel("Prime count")

plt.legend()

plt.tight_layout()

plt.savefig(FIG_DIR / "notebook29_reset_lane_counts.png")

plt.show()


## Reset pressure


In [ ]:
plt.figure(figsize=(14,5))

plt.plot(df["reset_pressure"])

plt.title("Notebook 29 — Reset Pressure")
plt.xlabel("Rolling window")
plt.ylabel("Pressure")

plt.tight_layout()

plt.savefig(FIG_DIR / "notebook29_reset_pressure.png")

plt.show()


## Sparse feature heatmap


In [ ]:
feature_cols = [
    "23",
    "29",
    "01",
    "gap_23_29",
    "gap_29_01",
    "reset_pressure",
    "boundary_pressure",
]

X = df[feature_cols].values

X = (X - X.mean(axis=0)) / X.std(axis=0)

plt.figure(figsize=(14,7))

plt.imshow(X.T, aspect="auto")

plt.yticks(range(len(feature_cols)), feature_cols)

plt.xlabel("Rolling window")
plt.ylabel("Feature")

plt.title("Notebook 29 — Sparse Feature Heatmap")

plt.colorbar(label="Scaled value")

plt.tight_layout()

plt.savefig(FIG_DIR / "notebook29_sparse_feature_heatmap.png")

plt.show()


## Sparse-event timeline


In [ ]:
plt.figure(figsize=(14,3))

event_idx = np.where(df["sparse_event"] == 1)[0]

plt.scatter(
    event_idx,
    np.ones_like(event_idx),
    s=30
)

plt.title("Notebook 29 — Sparse Event Timeline")
plt.xlabel("Rolling window")

plt.yticks([])

plt.tight_layout()

plt.savefig(FIG_DIR / "notebook29_sparse_event_timeline.png")

plt.show()


## Export outputs


In [ ]:
df.to_csv(
    RES_DIR / "notebook29_reset_feature_matrix.csv",
    index=False
)

events = df[df["sparse_event"] == 1]

events.to_csv(
    RES_DIR / "notebook29_sparse_events.csv",
    index=False
)

summary = {
    "num_windows": int(len(df)),
    "num_sparse_events": int(df["sparse_event"].sum()),
    "mean_reset_pressure": float(df["reset_pressure"].mean()),
    "max_reset_pressure": float(df["reset_pressure"].max()),
    "state_counts": (
        df["state"]
        .value_counts()
        .to_dict()
    )
}

with open(
    RES_DIR / "notebook29_reset_summary.json",
    "w"
) as f:

    json.dump(summary, f, indent=2)

summary
